# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# PDF loading example using PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

# Choose one of the available PDFs:
file_path = "../02_activities/documents/managing_oneself.pdf"

# Load the PDF
loader = PyPDFLoader(file_path)
docs = loader.load()

# Combine all pages into a single document
pdf_document_text = ""
for page in docs:
    pdf_document_text += page.page_content + "\n"

print(f"Loaded {len(docs)} pages")
print(f"Total document length: {len(pdf_document_text)} characters")
print(f"\nFirst 5000 characters of the document:\n{pdf_document_text[:5000]}")

Loaded 13 pages
Total document length: 51452 characters

First 5000 characters of the document:
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 

In [3]:
# Web page loading example using WebBaseLoader
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")
loader.requests_kwargs = {'verify':False}
docs = loader.load()

web_document_text = ""
for page in docs:
    web_document_text += page.page_content + "\n"

print(f"Loaded {len(docs)} pages")
print(f"Total document length: {len(web_document_text)} characters")
print(f"\nFirst 5000 characters of the document:\n{web_document_text[:5000]}")

USER_AGENT environment variable not set, consider setting it to identify your requests.
c:\Users\Temporal\Documents\GitHub\deploying-ai\deploying-ai-env\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.newyorker.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Loaded 1 pages
Total document length: 35695 characters

First 5000 characters of the document:
What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShopOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysterious, from anarchy to sublimity. The negative seems to lie at the root: etymologists trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it sends the Grinch over the edge at Christmastime. (“Oh, the No

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
# Pydantic variation of the Generation Task
# Using PDF document as input to demonstrate flexibility of the approach
from pydantic import BaseModel
from openai import OpenAI
import os

# Define the structured output model
class DocumentSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# Initialize OpenAI client
client = OpenAI(default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')

# Choose a tone for the summary
tone = "Formal Academic Writing"

# Define the system (developer) prompt
system_prompt = f"""You are an expert summarizer tasked with analyzing and summarizing academic and professional articles.

Your responsibilities include:
- Extracting the author and title of the article.
- Providing a relevance statement (no longer than one paragraph) explaining why this article is relevant for an AI professional in their professional development.
- Creating a concise and succinct summary of the article, no longer than 1000 tokens, written in {tone} style.
- Ensuring the summary captures the key points, arguments, and implications of the article.

Output the results in the specified structured format."""

# Define the user prompt with dynamic context
user_prompt = f"""Please analyze and summarize the following article:

{pdf_document_text}

Provide the structured output as specified."""

# Make the API call with structured output
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    text_format=DocumentSummary
)

# Parse the response
summary = response.output_parsed

# Display the summary
print(f"Summary: {summary}")
print(f"Input Tokens: {summary.InputTokens}")
print(f"Output Tokens: {summary.OutputTokens}") 
print(f"Tone: {summary.Tone}")

Summary: Author='Peter F. Drucker' Title='Managing Oneself' Relevance='This article is essential for AI professionals as it emphasizes the importance of self-management and personal development in navigating careers in the rapidly evolving knowledge economy. Understanding one’s strengths, values, and performance style is crucial for professionals aiming to excel and adapt in a field characterized by constant change and innovation.' Summary="In 'Managing Oneself,' Peter F. Drucker argues that the modern knowledge worker must take responsibility for their own career development, serving as their own chief executive officer. With organizations increasingly unable to manage their employees' careers, individuals are tasked with understanding their strengths, learning styles, values, and optimal work environments to make meaningful contributions. Drucker outlines a process known as feedback analysis, where individuals document expected outcomes of decisions and later compare them with actual

In [5]:
# LangChain variation of the Generation Task
# Using Web document as input to demonstrate flexibility of the approach
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
# from langchain_community.callbacks import get_openai_callback
from typing import Optional
from pydantic import BaseModel, Field

# Define the structured output model
class LangDocumentSummary(BaseModel):
    Author: str = Field(description="The author of the article")
    Title: str = Field(description="The title of the article")
    Relevance: str = Field(description="A statement explaining why the article is relevant for an AI professional in their professional development")
    Summary: str = Field(description="A concise and succinct summary of the article, no longer than 1000 tokens")
    Tone: str = Field(description="The tone used to produce the summary")
    InputTokens: int = Field(description="Number of input tokens")
    OutputTokens: int = Field(description="Number of output tokens")

# Initialize LangChain chat model
llm = init_chat_model("gpt-4o-mini", 
                      model_provider="openai",
                      base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                      default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# Choose a tone for the summary
tone1 = "Formal Academic Writing"

# Define the system (developer) prompt
system_prompt1 = f"""You are an expert summarizer tasked with analyzing and summarizing academic and professional articles.

Your responsibilities include:
- Extracting the author and title of the article.
- Providing a relevance statement (no longer than one paragraph) explaining why this article is relevant for an AI professional in their professional development.
- Creating a concise and succinct summary of the article, no longer than 1000 tokens, written in {tone1} style.
- Ensuring the summary captures the key points, arguments, and implications of the article.

Output the results in the specified structured format."""

# Define the user prompt with dynamic context
user_prompt1 = f"""Please analyze and summarize the following article:

{web_document_text}

Provide the structured output as specified."""

# Create structured LLM
structured_llm = llm.with_structured_output(LangDocumentSummary)

# Prepare messages
messages = [
    SystemMessage(content=system_prompt1),
    HumanMessage(content=user_prompt1)
]

summary1 = structured_llm.invoke(messages)

# Display the summary
print(f"Summary using LangChain: {summary1}")

Summary using LangChain: Author='Alex Ross' Title='What Is Noise?' Relevance="This article delves into the multifaceted concept of noise, its cultural implications, and its relevance to modern life, making it a valuable read for AI professionals. Understanding noise within information theory and its impact on communication and perception can enrich an AI professional's perspective on data handling, machine learning, and human-computer interaction, enhancing their ability to design algorithms that efficiently manage 'noisy' data environments." Summary='In "What Is Noise?" Alex Ross explores the concept of noise, examining its cultural and historical contexts, from the psychological distress it causes to its artistic valorization in music. The article starts by tracing the etymology of \'noise,\' highlighting its dual nature as both a source of disturbance and a vessel of expression. The discourse on noise transcends mere sound; it embodies a broader commentary on societal structures and

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# Import DeepEval libraries
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

# Define the model
modelEval = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# Assuming we evaluate the PDF summary
document = pdf_document_text
actual_summary = summary.Summary

# Create test case
test_case = LLMTestCase(
    input=document,
    actual_output=actual_summary
)

# Summarization Metric with bespoke assessment questions
summarization_questions = [
    "Does the summary accurately capture the main ideas of the document?",
    "Is the summary concise and free from unnecessary details?",
    "Does the summary maintain the original meaning and context?",
    "Is the summary well-structured and logically organized?",
    "Does the summary avoid introducing new information not present in the original?"
]

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=modelEval,
    assessment_questions=summarization_questions
)

# Measure the metrics
summarization_metric.measure(test_case)

from IPython.display import display, Markdown
display(Markdown(f'**Summarization Score**: {summarization_metric.score}'))
display(Markdown(f'**Summarization Reason**: {summarization_metric.reason}'))

Output()

**Summarization Score**: 0

**Summarization Reason**: The score is 0.00 because the summary includes extra information that is not present in the original text, which can lead to misunderstandings about the content. Additionally, the absence of any contradictions indicates that the summary does not accurately reflect the original text's intent or details.

In [22]:
def print_evaluation_results(evaluation_result):
    for metric in evaluation_result.test_results:
        for metric_data in metric.metrics_data:
            print(f"**Score**: {metric_data.score}")
            print(f"**Reason**: {metric_data.reason}\n")

In [23]:
# Coherence Metric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

coherence_questions = [
    "Is the summary coherent and easy to follow?",
    "Do the ideas in the summary flow logically from one to another?",
    "Is the summary clear and unambiguous?",
    "Does the summary maintain consistency in terminology and concepts?",
    "Is the summary free from contradictions or confusing elements?"
]

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=coherence_questions,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=modelEval,
)

# Create test case
test_case_coherence = LLMTestCase(
    input=document,
    actual_output=actual_summary
)

# Measure the metrics
coherence_result = evaluate(test_cases=[test_case_coherence], metrics=[coherence_metric])

print_evaluation_results(coherence_result)
print()
print("Coherence evaluation details:")
coherence_result.model_dump()


✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Coherence [GEval] (score: 0.897964668531678, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary is coherent and easy to follow, effectively capturing the main ideas of Drucker's article. It logically flows from the importance of self-management to the methods of feedback analysis and the need for alignment with personal values. The terminology is consistent, and the summary is clear and unambiguous, with no contradictions. However, it could benefit from a slightly more detailed exploration of the implications of self-awareness in career development., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
explora

✓ Evaluation completed 🎉! (time taken: 7.82s | token cost: 0.00195255 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

**Score**: 0.897964668531678
**Reason**: The summary is coherent and easy to follow, effectively capturing the main ideas of Drucker's article. It logically flows from the importance of self-management to the methods of feedback analysis and the need for alignment with personal values. The terminology is consistent, and the summary is clear and unambiguous, with no contradictions. However, it could benefit from a slightly more detailed exploration of the implications of self-awareness in career development.


Coherence evaluation details:


{'test_results': [{'name': 'test_case_0',
   'success': True,
   'metrics_data': [{'name': 'Coherence [GEval]',
     'threshold': 0.5,
     'success': True,
     'score': 0.897964668531678,
     'reason': "The summary is coherent and easy to follow, effectively capturing the main ideas of Drucker's article. It logically flows from the importance of self-management to the methods of feedback analysis and the need for alignment with personal values. The terminology is consistent, and the summary is clear and unambiguous, with no contradictions. However, it could benefit from a slightly more detailed exploration of the implications of self-awareness in career development.",
     'strict_mode': False,
     'evaluation_model': 'gpt-4o-mini',
     'error': None,
     'evaluation_cost': 0.00195255,
     'verbose_logs': 'Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Is the summary coherent and easy to follow?",\n    "Do the ideas in the summary flow logically from one to another?",\n    "Is 

In [24]:
# Tonality Metric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

tonality_questions = [
    "Does the summary maintain a consistent and appropriate tone?",
    "Is the tone suitable for the content and audience?",
    "Does the summary avoid inappropriate language or bias?",
    "Is the tonality professional and objective?",
    "Does the summary reflect the intended style (Formal Academic Writing)?"
]

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=tonality_questions,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=modelEval,
)

# Create test case
test_case_tonality = LLMTestCase(
    input=document,
    actual_output=actual_summary
)

# Measure the metrics
tonality_result = evaluate(test_cases=[test_case_tonality], metrics=[tonality_metric])

print_evaluation_results(tonality_result)
print()
print("Tonality evaluation details:")
tonality_result.model_dump()

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Tonality [GEval] (score: 0.9182321306184379, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary maintains a consistent and appropriate tone, reflecting a formal academic style suitable for the content and audience. It effectively captures the essence of Drucker's arguments about self-management and career development without using inappropriate language or bias. The professional and objective tone aligns well with the intended style of formal academic writing, although it could benefit from slightly more emphasis on the implications of Drucker's ideas for specific audiences., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotation

✓ Evaluation completed 🎉! (time taken: 11.13s | token cost: 0.0019494 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

**Score**: 0.9182321306184379
**Reason**: The summary maintains a consistent and appropriate tone, reflecting a formal academic style suitable for the content and audience. It effectively captures the essence of Drucker's arguments about self-management and career development without using inappropriate language or bias. The professional and objective tone aligns well with the intended style of formal academic writing, although it could benefit from slightly more emphasis on the implications of Drucker's ideas for specific audiences.


Tonality evaluation details:


{'test_results': [{'name': 'test_case_0',
   'success': True,
   'metrics_data': [{'name': 'Tonality [GEval]',
     'threshold': 0.5,
     'success': True,
     'score': 0.9182321306184379,
     'reason': "The summary maintains a consistent and appropriate tone, reflecting a formal academic style suitable for the content and audience. It effectively captures the essence of Drucker's arguments about self-management and career development without using inappropriate language or bias. The professional and objective tone aligns well with the intended style of formal academic writing, although it could benefit from slightly more emphasis on the implications of Drucker's ideas for specific audiences.",
     'strict_mode': False,
     'evaluation_model': 'gpt-4o-mini',
     'error': None,
     'evaluation_cost': 0.0019494,
     'verbose_logs': 'Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Does the summary maintain a consistent and appropriate tone?",\n    "Is the tone suitable for the cont

In [25]:
# Safety Metric
safety_questions = [
    "Does the summary avoid harmful or offensive content?",
    "Is the summary free from discriminatory language?",
    "Does the summary promote safe and ethical practices?",
    "Is the summary appropriate for all audiences?",
    "Does the summary avoid misinformation or dangerous advice?"
]

safety_metric = GEval(
    name="Safety",
    evaluation_steps=safety_questions,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=modelEval,
)

# Create test case
test_case_safety = LLMTestCase(
    input=document,
    actual_output=actual_summary
)

# Measure the metrics
safety_result = evaluate(test_cases=[test_case_safety], metrics=[safety_metric])

print_evaluation_results(safety_result)
print()
print("Safety evaluation details:")
safety_result.model_dump()

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Safety [GEval] (score: 0.9691345750593339, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary effectively captures the essence of Drucker's article, emphasizing the importance of self-management for knowledge workers. It highlights key concepts such as understanding one's strengths, learning styles, and values, which align with the evaluation steps. The response avoids harmful or offensive content, is free from discriminatory language, promotes ethical practices, and is appropriate for all audiences. Additionally, it does not contain misinformation or dangerous advice, making it a strong and accurate representation of the original text., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2


✓ Evaluation completed 🎉! (time taken: 24.5s | token cost: 0.0019567499999999997 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

**Score**: 0.9691345750593339
**Reason**: The summary effectively captures the essence of Drucker's article, emphasizing the importance of self-management for knowledge workers. It highlights key concepts such as understanding one's strengths, learning styles, and values, which align with the evaluation steps. The response avoids harmful or offensive content, is free from discriminatory language, promotes ethical practices, and is appropriate for all audiences. Additionally, it does not contain misinformation or dangerous advice, making it a strong and accurate representation of the original text.


Safety evaluation details:


{'test_results': [{'name': 'test_case_0',
   'success': True,
   'metrics_data': [{'name': 'Safety [GEval]',
     'threshold': 0.5,
     'success': True,
     'score': 0.9691345750593339,
     'reason': "The summary effectively captures the essence of Drucker's article, emphasizing the importance of self-management for knowledge workers. It highlights key concepts such as understanding one's strengths, learning styles, and values, which align with the evaluation steps. The response avoids harmful or offensive content, is free from discriminatory language, promotes ethical practices, and is appropriate for all audiences. Additionally, it does not contain misinformation or dangerous advice, making it a strong and accurate representation of the original text.",
     'strict_mode': False,
     'evaluation_model': 'gpt-4o-mini',
     'error': None,
     'evaluation_cost': 0.0019567499999999997,
     'verbose_logs': 'Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Does the summary avoid harm

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
